In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

tickets_raw = pd.read_csv('tickets.csv')
pd.read_csv('tickets.csv')

In [ ]:
teams = pd.read_csv('teams.csv')
pd.read_csv('teams.csv')

In [ ]:
print(f"Original tickets shape: {tickets_raw.shape} (Expected: 13 rows)")
tickets_clean = tickets_raw.drop_duplicates().copy()
print(f"Cleaned tickets shape: {tickets_clean.shape} (Expected: 12 rows)")
tickets_clean['resolution_hours'] = pd.to_numeric(tickets_clean['resolution_hours'])
tickets_clean['satisfaction'] = pd.to_numeric(tickets_clean['satisfaction'])
merged_df = pd.merge(tickets_clean, teams, on='team_id', how='left')
assert len(merged_df) == 12, f"Assertion Error: Expected 12 rows, got {len(merged_df)}"
assert merged_df['department'].isna().sum() == 0, "Assertion Error: Unmatched team_id found (NaN values in department)"
print("Merge assertions passed successfully: Exactly 12 rows and 0 unmatched keys.")

In [ ]:
print("\n--- 3. DERIVED FIELD & GROUPED ANALYSIS ---")
merged_df['breach_flag'] = (merged_df['resolution_hours'] > 24).astype(int)

dept_summary = merged_df.groupby('department').agg(
    total_tickets=('ticket_id', 'count'),
    breached_tickets=('breach_flag', 'sum')
).reset_index()

dept_summary['sla_breach_rate_pct'] = (dept_summary['breached_tickets'] / dept_summary['total_tickets']) * 100

print("\nDepartment Summary:")
print(dept_summary)

highest_team = team_summary.iloc[0]
print(f"\nTeam with Highest Breach Rate:")
print(f"Team ID: {highest_team['team_id']} ({highest_team['team']} - {highest_team['department']})")
print(f"Numerator (Breached Count): {int(highest_team['breached_tickets'])}")
print(f"Denominator (Total Tickets): {int(highest_team['total_tickets'])}")
print(f"SLA Breach Rate: {highest_team['sla_breach_rate_pct']:.2f}%")

In [ ]:
print("\n--- 4. VISUALIZATION (MONTHLY AVERAGE RESOLUTION HOURS) ---")
# Ensure month order is strictly: Jan -> Feb -> Mar
month_order = ['Jan', 'Feb', 'Mar']
merged_df['month'] = pd.Categorical(merged_df['month'], categories=month_order, ordered=True)

monthly_avg = merged_df.groupby('month', observed=False)['resolution_hours'].mean().reset_index()

# Plotting
plt.figure(figsize=(8, 5))
plt.bar(monthly_avg['month'], monthly_avg['resolution_hours'], color='skyblue', edgecolor='black')
plt.title('Average Resolution Hours by Month', fontsize=14, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Average Resolution Hours', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Save chart to outputs/
chart_path = 'outputs/python_chart.png'
plt.savefig(chart_path, bbox_inches='tight')
plt.close()
print(f"Chart successfully saved to {chart_path}")

print("\n--- 5. EXPORTING DATASETS ---")
merged_df.to_csv('outputs/clean_data.csv', index=False)
dept_summary.to_csv('outputs/python_summary.csv', index=False)
print("Files exported successfully to outputs/clean_data.csv and outputs/python_summary.csv")